# 1. Objective

Run the next image-embedding experiments without changing the stable baseline notebook. This notebook compares submission-safe image features, optional pretrained image embeddings, and reduced embedding variants under grouped cross-validation.

The notebook is internet-safe for Kaggle reruns: pretrained weights are used only when available locally or already cached. If they are unavailable, the embedding experiment is skipped cleanly.

In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageStat
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)

sns.set_theme(style='whitegrid', palette='viridis', rc={
    'figure.figsize': (11, 6),
    'axes.titlesize': 14,
    'axes.labelsize': 11,
})

DATA_DIR = Path('/kaggle/input/competitions/csiro-biomass')
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_ORDER = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
TARGET_WEIGHTS = {
    'Dry_Green_g': 0.1,
    'Dry_Dead_g': 0.1,
    'Dry_Clover_g': 0.1,
    'GDM_g': 0.2,
    'Dry_Total_g': 0.5,
}

# Kaggle scoring reruns usually disable internet. Keep this False unless you are in an interactive run.
ALLOW_MODEL_DOWNLOAD = False
# Optional: add an input dataset containing efficientnet_b0 weights and set this path.
LOCAL_EFFNET_B0_WEIGHTS = None

if not DATA_DIR.exists():
    raise FileNotFoundError(f'Expected Kaggle input at {DATA_DIR}')

print(f'Data directory: {DATA_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'ALLOW_MODEL_DOWNLOAD: {ALLOW_MODEL_DOWNLOAD}')

# 2. Data and Validation

## 2.1 Load Competition Files

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print(f'train: {train.shape} | train images: {train.image_path.nunique():,}')
print(f'test: {test.shape} | test images: {test.image_path.nunique():,}')
print(f'sample_submission: {sample_submission.shape}')
display(train.head())
display(test.head())

## 2.2 Metric and Grouped Folds

In [ ]:
def weighted_r2_score(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    weights = np.asarray(weights, dtype=float)
    weighted_mean = np.average(y_true, weights=weights)
    ss_res = np.sum(weights * np.square(y_true - y_pred))
    ss_tot = np.sum(weights * np.square(y_true - weighted_mean))
    return float(1.0 - ss_res / ss_tot) if ss_tot > 0 else 0.0

train = train.copy()
train['target_weight'] = train['target_name'].map(TARGET_WEIGHTS)

gkf = GroupKFold(n_splits=5)
train['fold'] = -1
for fold, (_, val_idx) in enumerate(gkf.split(train, groups=train['image_path']), start=1):
    train.loc[val_idx, 'fold'] = fold

display(pd.crosstab(train['fold'], train['target_name']))

# 3. Feature Frames

## 3.1 Image-Level Table

In [ ]:
meta_cols = ['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm', 'fold']
image_meta = train[meta_cols].drop_duplicates('image_path').copy()
image_targets = train.pivot_table(index='image_path', columns='target_name', values='target', aggfunc='first').reset_index()
image_train = image_meta.merge(image_targets, on='image_path', how='left')

test_image = test.drop_duplicates('image_path').copy()
for col in ['Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm']:
    if col not in test_image.columns:
        test_image[col] = np.nan

print(image_train.shape, test_image.shape)
display(image_train.head())

## 3.2 Metadata Features

In [ ]:
def add_metadata_features(df):
    out = df.copy()
    dates = pd.to_datetime(out.get('Sampling_Date'), errors='coerce')
    out['sample_month'] = dates.dt.month
    out['sample_dayofyear'] = dates.dt.dayofyear
    out['sample_year'] = dates.dt.year
    species = out['Species'].fillna('Unknown').astype(str)
    out['primary_species'] = species.str.split('_').str[0]
    out['species_count'] = np.where(species.eq('Unknown'), 0, species.str.count('_') + 1)
    out['ndvi_x_height'] = out['Pre_GSHH_NDVI'] * out['Height_Ave_cm']
    out['height_log1p'] = np.log1p(out['Height_Ave_cm'].clip(lower=0))
    return out

image_train = add_metadata_features(image_train)
test_image = add_metadata_features(test_image)

## 3.3 Color Features

In [ ]:
def extract_image_features(df, cache_name):
    cache_path = OUTPUT_DIR / cache_name
    if cache_path.exists():
        print(f'Loading cached features: {cache_path}')
        return pd.read_csv(cache_path)

    rows = []
    for rel_path in tqdm(df['image_path'].drop_duplicates().tolist(), desc=cache_name):
        path = DATA_DIR / rel_path
        row = {'image_path': rel_path}
        try:
            img = Image.open(path).convert('RGB')
            row['width'], row['height'] = img.size
            small = img.resize((256, 256))
            arr = np.asarray(small, dtype=np.float32) / 255.0
            stat = ImageStat.Stat(small)
            for idx, ch in enumerate(['r', 'g', 'b']):
                row[f'{ch}_mean'] = stat.mean[idx] / 255.0
                row[f'{ch}_std'] = stat.stddev[idx] / 255.0
                row[f'{ch}_p10'] = np.quantile(arr[:, :, idx], 0.10)
                row[f'{ch}_p50'] = np.quantile(arr[:, :, idx], 0.50)
                row[f'{ch}_p90'] = np.quantile(arr[:, :, idx], 0.90)
            red, green, blue = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
            row['brightness'] = arr.mean()
            row['contrast'] = arr.std()
            row['excess_green'] = np.mean(2 * green - red - blue)
            row['green_red_ratio'] = np.mean(green / (red + 1e-4))
            row['visible_ndvi_proxy'] = np.mean((green - red) / (green + red + 1e-4))
        except Exception as exc:
            row['image_error'] = str(exc)
        rows.append(row)

    features = pd.DataFrame(rows)
    features.to_csv(cache_path, index=False)
    print(f'Wrote {cache_path}')
    return features

train_img = extract_image_features(image_train, 'train_image_features.csv')
test_img = extract_image_features(test_image, 'test_image_features.csv')

image_train = image_train.merge(train_img, on='image_path', how='left')
test_image = test_image.merge(test_img, on='image_path', how='left')
print(f'Color feature columns: {len([c for c in train_img.columns if c != "image_path"])}')

# 4. Optional Pretrained Embeddings

This section does not require internet. It skips embeddings cleanly unless weights are already available or a local weights path is provided.

In [ ]:
def find_cached_effnet_weights():
    candidates = []
    if LOCAL_EFFNET_B0_WEIGHTS:
        candidates.append(Path(LOCAL_EFFNET_B0_WEIGHTS))
    candidates.extend(Path('/root/.cache/torch/hub/checkpoints').glob('efficientnet_b0*.pth'))
    candidates.extend(Path('/kaggle/input').glob('**/efficientnet_b0*.pth'))
    candidates = [p for p in candidates if p.exists()]
    return candidates[0] if candidates else None

def extract_effnet_b0_embeddings(df, cache_name, batch_size=16):
    cache_path = OUTPUT_DIR / cache_name
    if cache_path.exists():
        print(f'Loading cached embeddings: {cache_path}')
        return pd.read_csv(cache_path)

    try:
        import torch
        from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
    except Exception as exc:
        print(f'Skipping embeddings; torch/torchvision unavailable: {exc}')
        return pd.DataFrame({'image_path': df['image_path'].drop_duplicates().tolist()})

    weight_path = find_cached_effnet_weights()
    if weight_path is None and not ALLOW_MODEL_DOWNLOAD:
        print('Skipping embeddings; no local EfficientNet-B0 weights found and downloads are disabled.')
        return pd.DataFrame({'image_path': df['image_path'].drop_duplicates().tolist()})

    if weight_path is not None:
        print(f'Using local EfficientNet-B0 weights: {weight_path}')
        model = efficientnet_b0(weights=None)
        state = torch.load(weight_path, map_location='cpu')
        model.load_state_dict(state)
        preprocess = EfficientNet_B0_Weights.DEFAULT.transforms()
    else:
        print('Downloading EfficientNet-B0 weights because ALLOW_MODEL_DOWNLOAD=True.')
        weights = EfficientNet_B0_Weights.DEFAULT
        model = efficientnet_b0(weights=weights)
        preprocess = weights.transforms()

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.classifier = torch.nn.Identity()
    model = model.to(device).eval()

    rows = []
    unique_paths = df['image_path'].drop_duplicates().tolist()
    with torch.no_grad():
        for start in tqdm(range(0, len(unique_paths), batch_size), desc=cache_name):
            batch_paths = unique_paths[start:start + batch_size]
            images = []
            kept_paths = []
            for rel_path in batch_paths:
                try:
                    img = Image.open(DATA_DIR / rel_path).convert('RGB')
                    images.append(preprocess(img))
                    kept_paths.append(rel_path)
                except Exception as exc:
                    print(f'Skipping {rel_path}: {exc}')
            if not images:
                continue
            feats = model(torch.stack(images).to(device)).detach().cpu().numpy()
            for rel_path, vec in zip(kept_paths, feats):
                row = {'image_path': rel_path}
                row.update({f'effnet_b0_{i:04d}': float(v) for i, v in enumerate(vec)})
                rows.append(row)

    embeddings = pd.DataFrame(rows)
    embeddings.to_csv(cache_path, index=False)
    print(f'Wrote {cache_path}')
    return embeddings

train_emb = extract_effnet_b0_embeddings(image_train, 'train_effnet_b0_embeddings.csv')
test_emb = extract_effnet_b0_embeddings(test_image, 'test_effnet_b0_embeddings.csv')

image_train = image_train.merge(train_emb, on='image_path', how='left')
test_image = test_image.merge(test_emb, on='image_path', how='left')
embedding_cols = [c for c in image_train.columns if c.startswith('effnet_b0_')]
print(f'Embedding features available: {len(embedding_cols)}')

# 5. Experiment Matrix

## 5.1 Long-Format Dataset

In [ ]:
def to_long(df, include_targets=True):
    repeated = []
    for target_name in TARGET_ORDER:
        tmp = df.copy()
        tmp['target_name'] = target_name
        tmp['target_weight'] = TARGET_WEIGHTS[target_name]
        if include_targets:
            tmp['target'] = tmp[target_name]
        repeated.append(tmp)
    return pd.concat(repeated, ignore_index=True)

train_long = to_long(image_train, include_targets=True)
test_long = test.merge(test_image.drop(columns=[c for c in ['sample_id', 'target_name'] if c in test_image.columns]), on='image_path', how='left')
test_long['target_weight'] = test_long['target_name'].map(TARGET_WEIGHTS)

image_feature_base = [
    'width', 'height', 'brightness', 'contrast', 'excess_green', 'green_red_ratio', 'visible_ndvi_proxy',
    'r_mean', 'g_mean', 'b_mean', 'r_std', 'g_std', 'b_std',
    'r_p10', 'r_p50', 'r_p90', 'g_p10', 'g_p50', 'g_p90', 'b_p10', 'b_p50', 'b_p90',
]
metadata_feature_base = [
    'State', 'Species', 'primary_species', 'species_count', 'Pre_GSHH_NDVI', 'Height_Ave_cm',
    'ndvi_x_height', 'height_log1p', 'sample_month', 'sample_dayofyear', 'sample_year',
]

image_cols = ['target_name'] + [c for c in image_feature_base if c in train_long.columns]
metadata_cols = ['target_name'] + [c for c in metadata_feature_base if c in train_long.columns]
tabular_color_cols = list(dict.fromkeys(image_cols + metadata_cols))
embedding_cols = [c for c in train_long.columns if c.startswith('effnet_b0_')]

print(f'image/color features: {len(image_cols)}')
print(f'tabular/color features: {len(tabular_color_cols)}')
print(f'embedding features: {len(embedding_cols)}')

## 5.2 Reduced Embedding Features

In [ ]:
pca_cols = []
if embedding_cols:
    n_components = min(64, len(embedding_cols), image_train.shape[0] - 1)
    pca = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('pca', PCA(n_components=n_components, random_state=2026)),
    ])
    image_pca = pca.fit_transform(image_train[embedding_cols])
    test_pca = pca.transform(test_image[embedding_cols])
    pca_cols = [f'effnet_b0_pca_{i:02d}' for i in range(n_components)]
    image_train[pca_cols] = image_pca
    test_image[pca_cols] = test_pca
    train_long = to_long(image_train, include_targets=True)
    test_long = test.merge(test_image.drop(columns=[c for c in ['sample_id', 'target_name'] if c in test_image.columns]), on='image_path', how='left')
    test_long['target_weight'] = test_long['target_name'].map(TARGET_WEIGHTS)
    print(f'Created {len(pca_cols)} PCA embedding features')
else:
    print('No embeddings available; skipping PCA features.')

## 5.3 Models and Evaluation Helpers

In [ ]:
def make_preprocess(cols, scale_numeric=False):
    cat_cols = [c for c in ['target_name', 'State', 'Species', 'primary_species'] if c in cols]
    num_cols = [c for c in cols if c not in cat_cols]
    num_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale_numeric:
        num_steps.append(('scaler', StandardScaler()))
    return ColumnTransformer([
        ('num', Pipeline(num_steps), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=3)),
        ]), cat_cols),
    ])

def fit_with_weights(model, X, y, sample_weight):
    try:
        return model.fit(X, y, model__sample_weight=sample_weight)
    except TypeError:
        return model.fit(X, y)

def make_model(model_name, cols, seed=2026):
    if model_name == 'extra_trees':
        reg = ExtraTreesRegressor(
            n_estimators=400,
            min_samples_leaf=3,
            max_features=0.75,
            random_state=seed,
            n_jobs=-1,
        )
        return Pipeline([('preprocess', make_preprocess(cols)), ('model', reg)])
    if model_name == 'hgb':
        reg = HistGradientBoostingRegressor(
            max_iter=500,
            learning_rate=0.04,
            l2_regularization=0.05,
            max_leaf_nodes=31,
            random_state=seed,
        )
        return Pipeline([('preprocess', make_preprocess(cols)), ('model', reg)])
    raise ValueError(model_name)

def run_grouped_cv(experiment_name, cols, model_name='extra_trees'):
    cols = [c for c in cols if c in train_long.columns]
    parts = []
    print(f'Running {experiment_name}: {model_name}, {len(cols)} features')
    for fold in sorted(train_long['fold'].unique()):
        trn = train_long[train_long['fold'] != fold]
        val = train_long[train_long['fold'] == fold]
        model = make_model(model_name, cols, seed=2026 + int(fold))
        model = fit_with_weights(model, trn[cols], trn['target'], trn['target_weight'])
        pred = np.clip(model.predict(val[cols]), 0, None)
        keep_cols = [c for c in ['image_path', 'fold', 'State', 'primary_species', 'target_name', 'target', 'target_weight'] if c in val.columns]
        tmp = val[keep_cols].copy()
        tmp['prediction'] = pred
        tmp['experiment'] = experiment_name
        tmp['model'] = model_name
        parts.append(tmp)
    out = pd.concat(parts, ignore_index=True)
    out['error'] = out['prediction'] - out['target']
    out['abs_error'] = out['error'].abs()
    return out

def summarize_oof(oof):
    return (
        oof.groupby(['experiment', 'model'])
        .apply(lambda x: pd.Series({
            'weighted_r2': weighted_r2_score(x['target'], x['prediction'], x['target_weight']),
            'mae': mean_absolute_error(x['target'], x['prediction']),
            'bias': np.mean(x['error']),
        }))
        .reset_index()
        .sort_values('weighted_r2', ascending=False)
    )

# 6. Run Experiments

The oracle metadata experiments are diagnostic only when hidden test metadata is unavailable. Submission-safe experiments should rely on image-derived features plus `target_name`.

In [ ]:
experiments = {
    'image_color_only': image_cols,
    'tabular_color_oracle': tabular_color_cols,
}
if embedding_cols:
    experiments['embeddings_only'] = ['target_name'] + embedding_cols
    experiments['image_color_plus_embeddings'] = list(dict.fromkeys(image_cols + embedding_cols))
if pca_cols:
    experiments['embedding_pca_only'] = ['target_name'] + pca_cols
    experiments['image_color_plus_embedding_pca'] = list(dict.fromkeys(image_cols + pca_cols))

oof_parts = []
for experiment_name, cols in experiments.items():
    oof_parts.append(run_grouped_cv(experiment_name, cols, model_name='extra_trees'))

experiment_oof = pd.concat(oof_parts, ignore_index=True)
experiment_summary = summarize_oof(experiment_oof)
display(experiment_summary.round(5))

experiment_oof.to_csv(OUTPUT_DIR / 'embedding_experiment_oof.csv', index=False)
experiment_summary.to_csv(OUTPUT_DIR / 'embedding_experiment_summary.csv', index=False)
print(f'Wrote {OUTPUT_DIR / "embedding_experiment_oof.csv"}')
print(f'Wrote {OUTPUT_DIR / "embedding_experiment_summary.csv"}')

# 7. Segment Diagnostics

Focus on the segments that have been hardest in the baseline notebook, especially NSW high-biomass rows.

In [ ]:
best_experiment = experiment_summary.iloc[0]['experiment']
best_oof = experiment_oof[experiment_oof['experiment'].eq(best_experiment)].copy()

per_target = (
    best_oof.groupby('target_name')
    .apply(lambda x: pd.Series({
        'weighted_r2': weighted_r2_score(x['target'], x['prediction'], x['target_weight']),
        'mae': mean_absolute_error(x['target'], x['prediction']),
        'bias': np.mean(x['error']),
        'target_mean': x['target'].mean(),
    }))
    .reset_index()
    .sort_values('weighted_r2', ascending=False)
)

segment_error = (
    best_oof.groupby(['target_name', 'State'])
    .agg(rows=('target', 'size'), mae=('abs_error', 'mean'), bias=('error', 'mean'), target_mean=('target', 'mean'))
    .reset_index()
    .sort_values('mae', ascending=False)
)

print(f'Best experiment: {best_experiment}')
display(per_target.round(4))
display(segment_error.head(20).round(3))

per_target.to_csv(OUTPUT_DIR / 'embedding_experiment_per_target.csv', index=False)
segment_error.to_csv(OUTPUT_DIR / 'embedding_experiment_segment_error.csv', index=False)
print(f'Wrote {OUTPUT_DIR / "embedding_experiment_per_target.csv"}')
print(f'Wrote {OUTPUT_DIR / "embedding_experiment_segment_error.csv"}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_summary = experiment_summary.sort_values('weighted_r2', ascending=True)
sns.barplot(data=plot_summary, y='experiment', x='weighted_r2', hue='experiment', legend=False, ax=axes[0])
axes[0].set_title('Experiment Weighted R2')
axes[0].set_xlabel('Weighted R2')
axes[0].set_ylabel('')

sns.scatterplot(data=best_oof, x='target', y='prediction', hue='target_name', s=18, ax=axes[1])
limit = max(best_oof['target'].max(), best_oof['prediction'].max())
axes[1].plot([0, limit], [0, limit], color='black', linewidth=1)
axes[1].set_title('Best Experiment Predictions')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
plt.tight_layout()
plt.show()

# 8. Key Findings Template

Use this section after running the notebook on Kaggle. Keep the baseline notebook as the stable benchmark and promote only experiments that beat it under grouped CV and improve the hard segments.

- Best experiment: fill from `experiment_summary`.
- Compare against current stable baseline selected constrained weighted R2: `0.81160`.
- Promote an embedding approach only if it improves grouped CV and does not worsen `Dry_Total_g / NSW`, `GDM_g / NSW`, or `Dry_Green_g / NSW`.
- If pretrained embeddings are skipped because weights are unavailable, attach a Kaggle input dataset containing the model weights or run interactively with internet enabled only for feature generation.